# Evaluating taxonomy mappings

## Goal: measure the quality of proposed mappings against curated mappings

## Introduction

The risk mapper proposes cross-taxonomy mappings, either with the `SEMANTIC` method (embedding similarity) or the `INFERENCE` method (an LLM). These are suggestions that are meant to be reviewed by a human.

This notebook measures how good those suggestions are, by scoring them against the curated (human-reviewed) mappings that already ship in the repo. Running it gives a baseline that can be re-run after changes to quantify whether the mapping suggestions have improved.

We look at two things:

- **Retrieval**: for a given risk, did the mapper find the same related risks that a human curator had already linked it to? Reported as precision, recall and F1 on the risk pairs, ignoring direction.
- **Coverage**: for how many curated source risks did the mapper recover at least one of their curated mappings?

The example maps the IBM Risk Atlas onto the MIT AI Risk Repository. Only mappings marked `ManualMappingCuration` are used as ground truth, so the mapper is not scored against other machine-generated mappings.

One important detail: the MIT repository ships two kinds of entries. Its **domain** taxonomy holds actual risks, while its **causal** taxonomy holds classification factors (entity: AI / Human, intent: Intentional / Unintentional, timing: Pre- / Post-deployment). Those causal factors are not risks, and a risk-to-risk mapper is not meant to produce them, so we evaluate against the domain risks only.

In [1]:
from ai_atlas_nexus import AIAtlasNexus
from ai_atlas_nexus.blocks.risk_mapping import (
    evaluate_mappings,
    load_curated_mappings,
)
from ai_atlas_nexus.metadata_base import MappingMethod

## Load the risks

We load the IBM Risk Atlas risks and the MIT AI Risk Repository **domain** risks (the causal classification factors are excluded, as explained above).

In [2]:
nexus = AIAtlasNexus()

ibm_risks = nexus.get_all_risks(taxonomy="ibm-risk-atlas")
mit_risks = nexus.get_all_risks(taxonomy="mit-ai-risk-repository")

len(ibm_risks), len(mit_risks)

[2026-08-07 19:08:33:788] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: None


(99, 24)

## Generate the proposed mappings

We map each IBM risk onto the MIT domain risks with the `SEMANTIC` method. This method is deterministic and needs no LLM, so the baseline is stable and reproducible.

In [3]:
predicted = nexus.generate_proposed_mappings(
    new_risks=ibm_risks,
    existing_risks=mit_risks,
    inference_engine=None,
    new_prefix="ibm-risk-atlas",
    mapping_method=MappingMethod.SEMANTIC,
)

len(predicted)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[2026-08-07 19:09:37:333] - INFO - AIAtlasNexus - No match found for atlas-personal-information-in-prompt (best similarity 0.498)
[2026-08-07 19:09:39:588] - INFO - AIAtlasNexus - No match found for atlas-data-provenance (best similarity 0.489)
[2026-08-07 19:09:41:515] - INFO - AIAtlasNexus - No match found for atlas-copyright-infringement (best similarity 0.469)
[2026-08-07 19:09:43:308] - INFO - AIAtlasNexus - No match found for atlas-model-usage-rights (best similarity 0.406)
[2026-08-07 19:09:44:652] - INFO - AIAtlasNexus - No match found for atlas-improper-usage (best similarity 0.462)
[2026-08-07 19:09:45:18] - INFO - AIAtlasNexus - No match found for atlas-jailbreaking (best similarity 0.421)
[2026-08-07 19:09:46:36] - INFO - AIAtlasNexus - No match found for atlas-data-transfer (best similarity 0.477)
[2026-08-07 19:09:47:821] - INFO - AIAtlasNexus - No match found for atlas-temporal-gap (best similarity 0.482)


91

## Load the curated ground truth

`load_curated_mappings` reads the shipped mapping file, keeps only the manually curated rows, and drops any `noMatch` rows. We then keep only the mappings to actual MIT risks, so the causal classification factors are not counted.

In [4]:
ground_truth = load_curated_mappings(
    "mit-ai-risk-repository_ibm-risk-atlas.tsv"
)

mit_ids = {risk.id for risk in mit_risks}
ground_truth = [
    m for m in ground_truth if str(m.object_id).split(":")[-1] in mit_ids
]

len(ground_truth)

Not propagating value for 'mapping_date' because the slot is already set on individual records.


63

## Evaluate

`evaluate_mappings` compares the predicted mappings against the ground truth and returns the retrieval and coverage metrics.

In [5]:
import json

report = evaluate_mappings(predicted, ground_truth)
print(json.dumps(report, indent=2))

{
  "retrieval": {
    "precision": 0.3187,
    "recall": 0.4754,
    "f1": 0.3816,
    "n_predicted_pairs": 91,
    "n_ground_truth_pairs": 61,
    "n_matched_pairs": 29
  },
  "coverage": {
    "source_risk_coverage": 0.4754,
    "n_source_risks": 61,
    "n_source_risks_covered": 29
  }
}


## Baseline results

Running the above with the `SEMANTIC` method gives the following baseline:

| Metric | Value |
| --- | --- |
| Precision | 0.3187 |
| Recall | 0.4754 |
| F1 | 0.3816 |
| Source-risk coverage | 0.4754 |

From 91 proposed mappings, the mapper recovered 29 of the 61 curated risk-to-risk pairs, covering 29 of the 61 curated source risks.

Recall and coverage are the reliable signals here: from a single top match per risk, the embedding method recovers close to half of the curated risk-to-risk mappings. Precision reads lower because the mapper proposes a match for every risk while the curated set only records some pairs, so a proposed pair that a curator did not record still counts against precision even when it is plausible.